---
#### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---


# 4.2 — LLM Selection

---

**Scenario:** Your MLflow assistant has a solid eval dataset and an optimized prompt. But you've been running everything on a single model. Is it actually the best choice? Different LLMs vary in accuracy, latency, cost, and tone — and the only way to know which one fits your use case is to evaluate them head-to-head on your data.

**Objective:** Use the **MLflow AI Gateway** endpoints to route the same eval dataset through multiple LLMs, then compare their scores side by side to make a data-driven model selection.

## 💬 Product Check-in: Which Model Should We Use?

> **Sam (Product):** "We've been running on Gemini Flash this whole time. It's fast and cheap, but I keep hearing about GPT-4.1 mini and other models. Are we leaving quality on the table?"
>
> **You:** "Possibly. We picked Gemini Flash early because it was convenient, but we never actually compared it against alternatives on our eval data."
>
> **Sam:** "Can't we just look at benchmark leaderboards?"
>
> **You:** "Benchmarks tell you how models perform on generic tasks. Our assistant has specific requirements — MLflow domain knowledge, tool-calling accuracy, tone guidelines. The only way to know which model is best *for us* is to run our eval dataset through each one and compare."
>
> **Sam:** "That sounds like a lot of plumbing — different SDKs, different auth, different request formats."
>
> **You:** "That's where the AI Gateway pays off. Every model is behind a gateway endpoint with the same OpenAI-compatible interface. We swap the model name in one line and re-run the eval. Same predict function, same scorers, same dataset."
>
> **Sam:** "So we get an apples-to-apples comparison?"
>
> **You:** "Exactly. Same prompt, same questions, same judges. The only variable is the model."

---

Let's do exactly that. We'll:
1. Load the evaluation dataset
2. Define a predict function that routes through the gateway
3. Run evaluations across multiple models
4. Compare scores to pick the best model for our use case

---
## 0 — Environment Setup

In [ ]:
import os
from dotenv import load_dotenv
import mlflow

load_dotenv()

EXPERIMENT_NAME = os.environ.get("EXPERIMENT_4_NAME", "mlflow-agent-4")

if not os.getenv("MLFLOW_TRACKING_URI"):
    mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.openai.autolog()

---
## 1 — Load the Evaluation Dataset

We'll use the `full_eval_data` dataset built up over experiment 1. It contains MLflow API questions with `expected_facts`, `expected_response`, and `guidelines` — everything we need for a head-to-head model comparison.

In [ ]:
from mlflow.genai.datasets import get_dataset

full_eval_data = get_dataset(name="mlflow-agent-eval")

print(f"Loaded eval dataset: {len(full_eval_data.to_df())} records")
full_eval_data.to_df()[["inputs", "expectations"]].head(10)

---
## 2 — Gateway Client & Model Endpoints

The AI Gateway gives us a single OpenAI-compatible interface for every model. We point the OpenAI client at the gateway and swap the `model` parameter — that's it. No provider SDKs, no scattered API keys.

These endpoint names match what was configured in notebook 2.1. Update them to match your gateway setup.

In [ ]:
from openai import OpenAI

# Gateway client — all models accessible through one interface
gateway_client = OpenAI(
    base_url=f"{os.environ['MLFLOW_TRACKING_URI']}/gateway/mlflow/v1",
    api_key="",  # not needed, configured server-side
)

# ── Gateway endpoint names (from notebook 2.1) ────────────────────────
# Update these to match the endpoints configured in your AI Gateway
MODELS = {
    "gemini-3.1-free-tier": "Gemini 3.1 Flash Lite (Free)",
    # "gemini-3-flash-preview": "Gemini 3 Flash (Free)",
    # "gpt-4.1-mini":           "GPT-4.1 Mini",
    # "gpt-4.1-nano":           "GPT-4.1 Nano",
}

# Use a consistent judge model for scoring across all candidates
JUDGE_MODEL = "gemini-3.1-free-tier"

print(f"Models to compare: {len(MODELS)}")
for endpoint, label in MODELS.items():
    print(f"  - {endpoint} ({label})")

---
## 3 — Load the System Prompt

We'll use the same prompt for every model — the only variable in this experiment is the LLM itself.

In [ ]:
prompt = mlflow.genai.load_prompt("prompts:/mlflow-agent-system/@prod")

print("System prompt:")
print("---")
print(prompt.format()[:300])
print("---")

---
## 4 — Define the Predict Function

One predict function, parameterized by model name. The gateway handles routing to the right provider.

In [ ]:
def make_predict_fn(model_name: str):
    """Create a predict function bound to a specific gateway model endpoint."""
    def predict_fn(question: str) -> str:
        response = gateway_client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": prompt.format()},
                {"role": "user", "content": question},
            ],
            temperature=0,
            max_completion_tokens=512,
        )
        return response.choices[0].message.content
    return predict_fn

# Quick smoke test with the first model
first_model = list(MODELS.keys())[0]
test_fn = make_predict_fn(first_model)
print(f"Smoke test ({first_model}):")
print(test_fn("What is MLflow?")[:200])

---
## 5 — Run Evaluations Across Models

Same dataset, same prompt, same scorers, same judge — different model. Each eval run is tagged with the model name so we can filter and compare in the MLflow UI.

> **Note:** The judge model (`gateway:/`) also routes through the gateway, so judge calls get the same cost tracking and rate limiting as the candidate models.

In [ ]:
from mlflow.genai.scorers import Correctness, RelevanceToQuery, Guidelines

scorers = [
    Correctness(model=f"gateway:/{JUDGE_MODEL}"),
    RelevanceToQuery(model=f"gateway:/{JUDGE_MODEL}"),
    Guidelines(model=f"gateway:/{JUDGE_MODEL}"),
]

# Store results for each model
all_results = {}

for endpoint, label in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {label} ({endpoint})")
    print(f"{'='*60}")

    with mlflow.start_run(run_name=f"llm-selection-{endpoint}"):
        mlflow.log_param("model_endpoint", endpoint)
        mlflow.log_param("model_label", label)

        predict_fn = make_predict_fn(endpoint)

        results = mlflow.genai.evaluate(
            data=full_eval_data,
            predict_fn=predict_fn,
            scorers=scorers,
        )

        all_results[endpoint] = {
            "label": label,
            "metrics": results.metrics,
            "table": results.tables["eval_results_table"],
        }

        print(f"\nResults for {label}:")
        print(results.metrics)

print(f"\nCompleted evaluation for {len(all_results)} models.")

---
## 6 — Head-to-Head Comparison

Let's put the scores side by side. This is the table you'd show in a design review to justify a model choice.

In [ ]:
import pandas as pd

# Build comparison table
comparison_rows = []
for endpoint, data in all_results.items():
    row = {"Model": data["label"], "Endpoint": endpoint}
    row.update(data["metrics"])
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).set_index("Model")
comparison_df

In [ ]:
# Find the best model per scorer
metric_cols = [c for c in comparison_df.columns if c != "Endpoint"]
print("=== Best model per scorer ===")
for col in metric_cols:
    best = comparison_df[col].idxmax()
    print(f"  {col:<30} → {best} ({comparison_df[col].max():.3f})")

# Overall winner (average across all scorers)
comparison_df["avg_score"] = comparison_df[metric_cols].mean(axis=1)
winner = comparison_df["avg_score"].idxmax()
print(f"\n=== Overall winner (avg across scorers): {winner} ({comparison_df.loc[winner, 'avg_score']:.3f}) ===")

---
## 7 — Per-Question Breakdown

Aggregate scores can hide important differences. Let's look at which questions each model got right or wrong — this often reveals that one model excels at factual recall while another handles tone better.

In [ ]:
for endpoint, data in all_results.items():
    print(f"\n{'='*60}")
    print(f"{data['label']} — Per-question results")
    print(f"{'='*60}")
    table = data["table"]
    display(table)

---
## 8 — Check the MLflow UI

Open the MLflow UI and look at the experiment. You should see:

1. **One eval run per model** — each tagged with `model_endpoint` and `model_label` for easy filtering
2. **Identical scorers across runs** — same judge model, same criteria, apples-to-apples
3. **Trace details** — click into any run to see the actual model responses and judge reasoning

Use the **Compare** view to overlay runs and spot differences at a glance.

---
## Summary: What We Built

| Concept | How we used it |
|---|---|
| Product conversation | Sam's question → "are we using the right model?" |
| AI Gateway | Unified OpenAI-compatible interface to all models — swap one parameter |
| `get_dataset()` | Loaded the persistent eval dataset as the comparison benchmark |
| `make_predict_fn()` | Factory function parameterized by gateway endpoint name |
| `mlflow.genai.evaluate()` | Ran identical evals per model with consistent scorers and judge |
| Run tagging | Each run tagged with `model_endpoint` / `model_label` for filtering |
| Head-to-head comparison | Pandas DataFrame comparing all scorer metrics across models |
| Per-question breakdown | Revealed which questions each model handles differently |

### The pattern

```
Product feedback ("are we on the right model?")
  → Load the eval dataset (the comparison benchmark)
  → Load the system prompt (hold constant across models)
  → Point the gateway client at each endpoint
  → Run identical evals, tag each run with the model name
  → Compare aggregate scores + per-question breakdowns
  → Pick the model, justify with data
```

---

**Key insight:** The AI Gateway turns model selection from a plumbing problem into a one-line parameter change. Combined with a persistent eval dataset and consistent scorers, you can add a new model candidate in minutes — just create a gateway endpoint and re-run the notebook.